# Visualisasi Tahapan Lane Keeping Assist (LKA) Ackerman

Notebook ini memvisualisasikan langkah demi langkah pemrosesan citra dari modul LKA pada robot otonom Ackerman untuk mendapatkan nilai simpangan lajur (`lane_error`). 

Anda dapat menjalankan notebook ini langsung di **Google Colab** tanpa memerlukan file eksternal karena notebook ini akan menghasilkan gambar jalan sintetis secara mandiri menggunakan NumPy dan OpenCV.

## 1. Import Library & Setup Fungsi Pembantu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Fungsi untuk menampilkan gambar dengan Matplotlib
def show_image(img, title="Gambar", cmap=None):
    plt.figure(figsize=(8, 6))
    if len(img.shape) == 3:
        # OpenCV menggunakan BGR, Matplotlib menggunakan RGB
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.show()

# Fungsi untuk mendapatkan koordinat X perpotongan garis pada tinggi Y target
def get_x_intercept(pts, y_target):
    if len(pts) < 2:
        return -1
    # Fit line menggunakan metode Least Squares (L2)
    [vx, vy, x0, y0] = cv2.fitLine(np.array(pts), cv2.DIST_L2, 0, 0.01, 0.01)
    if vy == 0:
        return int(x0[0])
    return int(np.round(x0[0] + (vx[0] / vy[0]) * (y_target - y0[0])))

## 2. Pembuatan Gambar Jalan Sintetis (Input Kamera)

Kita akan membuat representasi citra $640 \times 480$ piksel yang mensimulasikan aspal gelap dengan dua garis marka putih miring.

In [ ]:
# Membuat aspal abu-abu gelap 640x480
input_frame = np.zeros((480, 640, 3), dtype=np.uint8)
input_frame[:, :] = [45, 45, 45]  # Nilai BGR aspal

# Menggambar marka jalan kiri (warna putih)
cv2.line(input_frame, (200, 120), (60, 480), (240, 240, 240), 12)

# Menggambar marka jalan kanan (warna putih)
cv2.line(input_frame, (440, 120), (580, 480), (240, 240, 240), 12)

show_image(input_frame, "Input Kamera (Simulasi Jalan Fisik)")

## 3. Tahap 1: Region of Interest (ROI) Masking

Untuk menghindari mendeteksi objek di luar jalan (seperti langit atau dinding laboratorium), kita membatasi area pemrosesan hanya pada poligon jalan (tinggi $y = 120$ hingga $480$).

In [ ]:
height, width = input_frame.shape[:2]
roi_mask = np.zeros((height, width), dtype=np.uint8)

# Titik-titik poligon ROI sesuai lanekeepingassist.cpp
roi_pts = np.array([[
    [0, 480],
    [640, 480],
    [640, 120],
    [0, 120]
]], dtype=np.int32)

cv2.fillConvexPoly(roi_mask, roi_pts, 255)

# Aplikasikan ROI pada gambar
roi_output = cv2.bitwise_and(input_frame, input_frame, mask=roi_mask)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(roi_mask, cmap='gray')
plt.title("Mask ROI", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(roi_output, cv2.COLOR_BGR2RGB))
plt.title("Input setelah ROI", fontsize=12)
plt.axis('off')
plt.show()

## 4. Tahap 2: Segmentasi Warna Putih HLS

Mengonversi gambar BGR ke ruang warna HLS (Hue, Lightness, Saturation) untuk meny menyaring marka putih dengan rentang pencahayaan (*Lightness*) yang tinggi ($183$ s.d. $255$).

In [ ]:
hls = cv2.cvtColor(input_frame, cv2.COLOR_BGR2HLS)

# Range warna putih pada marka jalan
lower_white = np.array([0, 183, 0])
upper_white = np.array([180, 255, 255])

white_mask = cv2.inRange(hls, lower_white, upper_white)
white_mask_roi = cv2.bitwise_and(white_mask, roi_mask)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(white_mask, cmap='gray')
plt.title("Filter HLS Putih", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(white_mask_roi, cmap='gray')
plt.title("HLS Putih + Mask ROI", fontsize=12)
plt.axis('off')
plt.show()

## 5. Tahap 3: Deteksi Tepi (Canny) & Hough Transform

Deteksi tepi Canny digunakan untuk mengekstrak bentuk garis, lalu Hough Transform digunakan untuk mendeteksi vektor segmen garis.

In [ ]:
# 1. Canny Edge
edges = cv2.Canny(white_mask_roi, 50, 150)

# 2. Hough Line Transform
lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=40, maxLineGap=20)

# Visualisasi Garis Hough di atas gambar asli
hough_visual = input_frame.copy()
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(hough_visual, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Warna merah untuk Hough Lines

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(edges, cmap='gray')
plt.title("Deteksi Tepi Canny", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(hough_visual, cv2.COLOR_BGR2RGB))
plt.title("Deteksi Segmen Garis Hough", fontsize=12)
plt.axis('off')
plt.show()

## 6. Tahap 4: Pemisahan Lajur Kiri & Kanan (Slopes Classification)

Mengklasifikasikan titik-titik garis berdasarkan kemiringannya (*slope*):
*   Marka Kiri memiliki slope negatif (antara $-3.0$ dan $-0.3$).
*   Marka Kanan memiliki slope positif (antara $0.3$ dan $3.0$).

In [ ]:
left_pts = []
right_pts = []

if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        slope = (y2 - y1) / (x2 - x1 + 1e-6)
        if -3.0 < slope < -0.3:
            left_pts.append((x1, y1))
            left_pts.append((x2, y2))
        elif 0.3 < slope < 3.0:
            right_pts.append((x1, y1))
            right_pts.append((x2, y2))

print(f"Jumlah titik marka kiri terdeteksi: {len(left_pts)}")
print(f"Jumlah titik marka kanan terdeteksi: {len(right_pts)}")

## 7. Tahap 5: Perhitungan Lane Error (Jarak Pandang $y = 300$ & Deadzone)

Kita mencari perpotongan garis kiri dan kanan pada ketinggian `target_y = 300`. 
*   `smoothed_lane_center` adalah rata-rata perpotongan tersebut.
*   `lane_error` adalah selisih `smoothed_lane_center` terhadap pusat sensor kamera (`center_x = 320`).
*   `deadzone` sebesar `30` piksel diterapkan untuk meredam jitter/noise kecil.

In [ ]:
target_y = 300
center_x = 320

# 1. Dapatkan X Intercept pada y = 300
left_x = get_x_intercept(left_pts, target_y)
right_x = get_x_intercept(right_pts, target_y)

# 2. Rata-rata pusat jalan
raw_lane_center = (left_x + right_x) / 2.0

# 3. Perhitungan simpangan
error = raw_lane_center - center_x
deadzone = 30.0

if abs(error) < deadzone:
    final_error = 0.0
else:
    if error > 0:
        final_error = error - deadzone
    else:
        final_error = error + deadzone

print("========== HASIL PERHITUNGAN LKA ==========")
print(f"Titik X Kiri di Y=300       : {left_x} px")
print(f"Titik X Kanan di Y=300      : {right_x} px")
print(f"Pusat Lajur (Lane Center)   : {raw_lane_center} px")
print(f"Pusat Sensor (Center X)     : {center_x} px")
print(f"Simpangan Kasar (Raw Error) : {error} px")
print(f"Simpangan Final (Deadzone)  : {final_error} px")

## 8. Visualisasi Hasil Akhir untuk Buku/Laporan

Visualisasi lengkap yang memperlihatkan garis bidik *lookahead* ($y = 300$), pusat referensi sensor, dan titik estimasi tengah jalan.

In [ ]:
output_frame = input_frame.copy()

# 1. Gambar Garis Lookahead (Y = 300) berwarna biru tipis
cv2.line(output_frame, (0, target_y), (640, target_y), (255, 100, 0), 1)

# 2. Gambar Garis Tengah Kamera (Center X = 320) vertikal berwarna biru
cv2.line(output_frame, (center_x, 0), (center_x, 480), (255, 0, 0), 1)

# 3. Plot Titik tengah jalan teoretis (Hijau)
cv2.circle(output_frame, (int(raw_lane_center), target_y), 8, (0, 255, 0), -1)

# 4. Plot Titik tengah sensor (Merah)
cv2.circle(output_frame, (center_x, target_y), 6, (0, 0, 255), -1)

# 5. Gambar tanda panah deviasi (Error)
cv2.arrowedLine(output_frame, (center_x, target_y), (int(raw_lane_center), target_y), (0, 255, 255), 2, tipLength=0.2)

# Tampilkan visualisasi
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB))

# Menambahkan Teks Informasi
plt.text(20, 40, f"Lookahead Target Y: {target_y} px", color='white', fontsize=12, bbox=dict(facecolor='black', alpha=0.6))
plt.text(20, 80, f"Raw Lane Error: {error:+.1f} px", color='cyan', fontsize=12, bbox=dict(facecolor='black', alpha=0.6))
plt.text(20, 120, f"Final Error (Deadzone 30px): {final_error:+.1f} px", color='yellow', fontsize=12, bbox=dict(facecolor='black', alpha=0.6))

plt.title("Visualisasi Akhir Pencarian Lane Error LKA", fontsize=14)
plt.axis('off')
plt.show()